# 第13章（一）：用户自定义数据类型 User-Defined Data Types

## Cambridge A-Level Computer Science (9618)

---

### 本节学习目标 Learning Objectives

1. 理解**为什么**我们需要用户自定义数据类型
2. 掌握**非复合类型 Non-composite types**：枚举类型、指针类型
3. 掌握**复合类型 Composite types**：记录类型、集合类型、类/对象类型
4. 学会根据实际场景选择合适的数据类型

---

### 为什么需要用户自定义数据类型？Why Do We Need User-Defined Data Types?

编程语言自带的**内置数据类型 (built-in data types)** 如 `INTEGER`、`REAL`、`STRING`、`BOOLEAN` 虽然强大，但在实际开发中往往不够用。

**核心原因：**
- **语义清晰**：`Day = Monday` 比 `Day = 1` 更易读、更不容易出错
- **类型安全**：编译器能帮你检查类型错误，防止把"颜色"赋值给"星期几"
- **结构化组织**：现实世界的数据（如学生信息）包含多种不同类型的字段
- **代码复用**：定义一次，到处使用

> **Cambridge考试重点**：你需要能解释用户自定义数据类型的**定义**和**必要性**。

In [ ]:
# 环境配置
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from enum import Enum, auto
from dataclasses import dataclass
from typing import Optional, Set
import ctypes

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

print('环境配置完成！')

---

## 一、非复合类型 Non-Composite Types

非复合类型是指**不由其他数据类型组合而成**的用户自定义类型。它是一个**单一的值**，不能再拆分成更小的部分。

### 1.1 枚举类型 Enumerated Type

#### 是什么？What is it?

枚举类型定义了一个**有限的、有序的值的列表**。每个值都有一个**名称**和一个隐含的**序号（ordinal）**。

**伪代码定义（Cambridge格式）：**
```
TYPE Season = (Spring, Summer, Autumn, Winter)
TYPE Day = (Mon, Tue, Wed, Thu, Fri, Sat, Sun)
```

#### 为什么需要它？Why do we need it?

| 不使用枚举 | 使用枚举 |
|:---|:---|
| `traffic_light = 1` | `traffic_light = Color.RED` |
| 1代表什么？谁记得住？ | 含义一目了然 |
| 可能赋值为999（无效值） | 只能是预定义的值 |
| 无法利用编译器检查 | 编译器能检查类型匹配 |

In [ ]:
# === 枚举类型 Enumerated Type 演示 ===

# --- 方式1：使用 enum.Enum ---
class Season(Enum):
    SPRING = auto()  # auto() 自动分配序号
    SUMMER = auto()
    AUTUMN = auto()
    WINTER = auto()

class TrafficLight(Enum):
    RED = 1
    AMBER = 2
    GREEN = 3

# --- 使用枚举 ---
current_season = Season.SUMMER
print(f"当前季节: {current_season}")
print(f"名称 name: {current_season.name}")
print(f"值 value: {current_season.value}")
print()

# --- 枚举是有序的，可以遍历 ---
print("所有季节 All seasons:")
for s in Season:
    print(f"  {s.name} -> ordinal {s.value}")
print()

# --- 类型安全演示 ---
print("=== 类型安全 Type Safety ===")
print(f"Season.SPRING == Season.SPRING: {Season.SPRING == Season.SPRING}")
print(f"Season.SPRING == TrafficLight.RED: {Season.SPRING == TrafficLight.RED}")
print("  --> 不同枚举类型之间比较返回 False，防止逻辑错误！")

In [ ]:
# === 枚举的实际应用：扑克牌 ===

class Suit(Enum):
    """花色"""
    HEARTS = '\u2665'    # 红心
    DIAMONDS = '\u2666'  # 方块
    CLUBS = '\u2663'     # 梅花
    SPADES = '\u2660'    # 黑桃

class Rank(Enum):
    """牌面值"""
    ACE = 1
    TWO = 2
    THREE = 3
    FOUR = 4
    FIVE = 5
    SIX = 6
    SEVEN = 7
    EIGHT = 8
    NINE = 9
    TEN = 10
    JACK = 11
    QUEEN = 12
    KING = 13

# 展示所有花色
print("扑克牌花色 Card Suits:")
for suit in Suit:
    print(f"  {suit.name}: {suit.value}", end="  ")
print("\n")

# 生成一手牌
import random
hand = [(random.choice(list(Rank)), random.choice(list(Suit))) for _ in range(5)]
print("你的手牌 Your hand:")
for rank, suit in hand:
    print(f"  {rank.name} of {suit.name} {suit.value}")

### 1.2 指针类型 Pointer Type

#### 是什么？What is it?

指针是一个变量，它**存储的不是数据本身，而是另一个变量在内存中的地址**。

**伪代码定义：**
```
TYPE TIntPointer = ^INTEGER
```

#### 为什么需要指针？Why do we need pointers?

1. **动态数据结构**：链表、树、图等数据结构必须用指针连接节点
2. **内存效率**：传递大对象时，传地址（4或8字节）比复制整个对象快得多
3. **动态内存分配**：在运行时（而非编译时）决定需要多少内存
4. **间接访问**：通过一个变量间接修改另一个变量的值

#### 关键概念

| 术语 | 英文 | 含义 |
|:---|:---|:---|
| 引用 | Referencing | 获取变量的地址 |
| 解引用 | Dereferencing | 通过地址访问变量的值 |
| 空指针 | Null pointer | 不指向任何地址的指针 |
| 悬空指针 | Dangling pointer | 指向已释放内存的指针（危险！） |

In [ ]:
# === 指针类型 Pointer Type 演示 ===

# Python没有显式指针，但我们可以用id()展示内存地址的概念

# --- 1. 内存地址的概念 ---
x = 42
y = x  # y 和 x 指向同一个对象

print("=== 内存地址演示 Memory Address Demo ===")
print(f"x = {x}, 内存地址 id(x) = {id(x)}")
print(f"y = {y}, 内存地址 id(y) = {id(y)}")
print(f"x 和 y 指向同一地址: {id(x) == id(y)}")
print()

# 修改 y 后，它指向新地址
y = 100
print(f"修改 y = {y} 后:")
print(f"x 地址: {id(x)}, y 地址: {id(y)}")
print(f"现在指向不同地址: {id(x) != id(y)}")
print()

# --- 2. 列表的引用行为（类似指针） ---
print("=== 列表的引用行为（类似指针）===")
list_a = [1, 2, 3]
list_b = list_a  # list_b 指向同一个列表！

print(f"list_a = {list_a}, id = {id(list_a)}")
print(f"list_b = {list_b}, id = {id(list_b)}")
print(f"指向同一对象: {list_a is list_b}")

list_b.append(999)
print(f"\n修改 list_b 后:")
print(f"list_a = {list_a}  <-- list_a 也变了！因为它们是同一个对象")
print(f"list_b = {list_b}")

In [ ]:
# === 用Python模拟指针和链表 ===

class Node:
    """链表节点 - 展示指针的核心应用"""
    def __init__(self, data):
        self.data = data
        self.next = None  # 指针，指向下一个节点（或None = 空指针）

class LinkedList:
    def __init__(self):
        self.head = None  # 头指针
    
    def append(self, data):
        new_node = Node(data)
        if self.head is None:
            self.head = new_node
            return
        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node  # 让最后一个节点的指针指向新节点
    
    def display(self):
        current = self.head
        result = []
        while current is not None:
            result.append(f"[{current.data}|*]")
            current = current.next
        result.append("[NULL]")
        print(" -> ".join(result))

# 创建链表
ll = LinkedList()
for item in ["Alice", "Bob", "Charlie", "Diana"]:
    ll.append(item)

print("链表结构 Linked List Structure:")
ll.display()
print()
print("说明：[数据|*] 中的 * 代表指针，指向下一个节点")
print("      NULL 代表空指针，表示链表结束")

In [ ]:
# === 可视化：指针与链表的内存示意图 ===

fig, ax = plt.subplots(1, 1, figsize=(14, 5))

nodes = ["Alice", "Bob", "Charlie", "Diana"]
addresses = ["0x1000", "0x1020", "0x1040", "0x1060"]

for i, (name, addr) in enumerate(zip(nodes, addresses)):
    x = i * 3.2
    # 数据部分
    rect_data = mpatches.FancyBboxPatch((x, 1), 1.5, 1.2, 
                                         boxstyle="round,pad=0.1",
                                         facecolor='#3498db', edgecolor='black', linewidth=2)
    ax.add_patch(rect_data)
    ax.text(x + 0.75, 1.6, name, ha='center', va='center', fontsize=12, 
            fontweight='bold', color='white')
    
    # 指针部分
    rect_ptr = mpatches.FancyBboxPatch((x + 1.5, 1), 0.8, 1.2,
                                        boxstyle="round,pad=0.1",
                                        facecolor='#e74c3c', edgecolor='black', linewidth=2)
    ax.add_patch(rect_ptr)
    
    if i < len(nodes) - 1:
        ax.text(x + 1.9, 1.6, addresses[i+1], ha='center', va='center', fontsize=8, color='white')
        # 箭头
        ax.annotate('', xy=((i+1)*3.2, 1.6), xytext=(x+2.3, 1.6),
                    arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2.5))
    else:
        ax.text(x + 1.9, 1.6, 'NULL', ha='center', va='center', fontsize=10, 
                fontweight='bold', color='white')
    
    # 地址标签
    ax.text(x + 1.15, 0.6, f"Address: {addr}", ha='center', va='center', fontsize=9, color='gray')

# Head pointer
ax.annotate('', xy=(0, 1.6), xytext=(-1.2, 1.6),
            arrowprops=dict(arrowstyle='->', color='green', lw=3))
ax.text(-1.8, 1.6, 'HEAD', ha='center', va='center', fontsize=13, fontweight='bold', color='green')

ax.set_xlim(-2.5, 14)
ax.set_ylim(0, 3.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('链表的内存布局 Memory Layout of a Linked List\n蓝色=数据 Data | 红色=指针 Pointer', 
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 二、复合类型 Composite Types

复合类型是**由多个现有数据类型组合而成**的类型。它可以包含不同类型的多个元素。

### 2.1 记录类型 Record Type

#### 是什么？What is it?

记录类型是一种**将不同类型的数据项组合在一起**的数据结构。每个数据项称为一个**字段 (field)**。

**伪代码定义：**
```
TYPE Student
    DECLARE Name : STRING
    DECLARE Age : INTEGER
    DECLARE Grade : CHAR
    DECLARE Enrolled : BOOLEAN
ENDTYPE
```

#### 为什么需要记录类型？

想象你在管理一个学校系统：
- 没有记录类型：你需要**4个独立的数组** `names[]`, `ages[]`, `grades[]`, `enrolled[]`，并通过**相同的索引**来关联它们 -> 容易出错！
- 有了记录类型：一个 `Student` 变量就包含了所有相关信息 -> 清晰、安全、易维护！

In [ ]:
# === 记录类型 Record Type 演示 ===

# Python中用 dataclass 模拟记录类型（最接近Cambridge伪代码的方式）

@dataclass
class Student:
    """学生记录 Student Record"""
    name: str
    age: int
    grade: str
    enrolled: bool

@dataclass
class Book:
    """图书记录 Book Record"""
    title: str
    author: str
    isbn: str
    price: float
    in_stock: bool

# --- 创建记录 ---
s1 = Student(name="Zhang Wei", age=17, grade="A", enrolled=True)
s2 = Student(name="Li Ming", age=16, grade="B", enrolled=True)
s3 = Student(name="Wang Fang", age=17, grade="A*", enrolled=False)

print("=== 学生记录 Student Records ===")
for s in [s1, s2, s3]:
    status = "在读" if s.enrolled else "未注册"
    print(f"  {s.name}, 年龄 {s.age}, 成绩 {s.grade}, 状态: {status}")

print(f"\n访问单个字段: s1.name = '{s1.name}', s1.grade = '{s1.grade}'")

# --- 记录数组 ---
print("\n=== 记录数组 Array of Records ===")
books = [
    Book("Python编程", "Guido", "978-0-13-4", 59.99, True),
    Book("算法导论", "CLRS", "978-0-26-2", 89.99, True),
    Book("数据库系统", "Silberschatz", "978-0-07-3", 75.50, False),
]
for b in books:
    stock = "有货" if b.in_stock else "缺货"
    print(f"  《{b.title}》 by {b.author} | {b.isbn} | ${b.price} | {stock}")

### 2.2 集合类型 Set Type

#### 是什么？What is it?

集合是一种**无序的、不包含重复元素**的数据集合。

**伪代码定义：**
```
TYPE Letters = SET OF CHAR
DECLARE Vowels : Letters
Vowels <- {'a', 'e', 'i', 'o', 'u'}
```

#### 为什么需要集合？

1. **快速查找**：检查元素是否存在是 O(1) 操作（哈希实现），而列表是 O(n)
2. **自动去重**：添加重复元素不会改变集合
3. **数学运算**：支持并集、交集、差集等操作，非常适合处理分类和分组问题

In [ ]:
# === 集合类型 Set Type 演示 ===

# --- 基本操作 ---
vowels = {'a', 'e', 'i', 'o', 'u'}
consonants = {'b', 'c', 'd', 'f', 'g', 'h', 'j', 'k', 'l', 'm',
              'n', 'p', 'q', 'r', 's', 't', 'v', 'w', 'x', 'y', 'z'}

print(f"元音集合 Vowels: {vowels}")
print(f"'a' in vowels: {('a' in vowels)}")
print(f"'b' in vowels: {('b' in vowels)}")
print()

# --- 自动去重 ---
numbers = {1, 2, 3, 2, 1, 4, 3, 5}
print(f"自动去重: {{1,2,3,2,1,4,3,5}} -> {numbers}")
print()

# --- 集合运算 ---
cs_students = {"Alice", "Bob", "Charlie", "Diana", "Eve"}
math_students = {"Bob", "Diana", "Frank", "Grace"}

print("=== 集合运算 Set Operations ===")
print(f"CS学生: {cs_students}")
print(f"数学学生: {math_students}")
print(f"并集 Union (两门课都选的): {cs_students | math_students}")
print(f"交集 Intersection (同时选两门课): {cs_students & math_students}")
print(f"差集 Difference (只选CS): {cs_students - math_students}")
print(f"对称差集 Symmetric Diff (只选一门): {cs_students ^ math_students}")

In [ ]:
# === 可视化：集合运算韦恩图 Venn Diagram ===

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def draw_venn(ax, title, highlight='none'):
    circle1 = plt.Circle((0.35, 0.5), 0.3, alpha=0.3, 
                          color='blue' if highlight in ('left', 'union', 'intersection', 'sym') else 'lightblue')
    circle2 = plt.Circle((0.65, 0.5), 0.3, alpha=0.3, 
                          color='red' if highlight in ('right', 'union', 'intersection', 'sym') else 'lightyellow')
    ax.add_patch(circle1)
    ax.add_patch(circle2)
    
    if highlight == 'intersection':
        from matplotlib.patches import Wedge
        ax.text(0.5, 0.5, 'A \u2229 B', ha='center', va='center', fontsize=16, fontweight='bold')
    elif highlight == 'union':
        ax.text(0.5, 0.5, 'A \u222A B', ha='center', va='center', fontsize=16, fontweight='bold')
    elif highlight == 'left':
        ax.text(0.5, 0.5, 'A - B', ha='center', va='center', fontsize=16, fontweight='bold')
    
    ax.text(0.2, 0.5, 'CS', ha='center', va='center', fontsize=14)
    ax.text(0.8, 0.5, 'Math', ha='center', va='center', fontsize=14)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

draw_venn(axes[0], '并集 Union (A \u222A B)', 'union')
draw_venn(axes[1], '交集 Intersection (A \u2229 B)', 'intersection')
draw_venn(axes[2], '差集 Difference (A - B)', 'left')

plt.suptitle('集合运算可视化 Set Operations Visualization', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.3 类/对象类型 Class/Object Type

#### 是什么？What is it?

类（Class）是一种**结合了数据（属性 attributes）和行为（方法 methods）的用户自定义类型**。对象（Object）是类的**实例 (instance)**。

#### 类与记录的区别

| 特征 | 记录 Record | 类 Class |
|:---|:---|:---|
| 包含数据 | 是 | 是 |
| 包含方法 | 否 | **是** |
| 继承 | 不支持 | **支持** |
| 封装 | 不支持 | **支持** |
| 多态 | 不支持 | **支持** |

#### 为什么需要类？

类是**面向对象编程 (OOP)** 的基础。它让我们能把**数据和操作数据的代码捆绑在一起**，使代码更加模块化、可复用、可维护。

In [ ]:
# === 类/对象类型 Class/Object Type 演示 ===

class BankAccount:
    """银行账户类 - 展示封装和方法"""
    
    def __init__(self, owner: str, balance: float = 0.0):
        self.__owner = owner        # 私有属性（封装）
        self.__balance = balance    # 私有属性
        self.__transactions = []    # 交易记录
    
    def deposit(self, amount: float) -> bool:
        """存款 Deposit"""
        if amount > 0:
            self.__balance += amount
            self.__transactions.append(f"+{amount:.2f}")
            return True
        return False
    
    def withdraw(self, amount: float) -> bool:
        """取款 Withdraw"""
        if 0 < amount <= self.__balance:
            self.__balance -= amount
            self.__transactions.append(f"-{amount:.2f}")
            return True
        return False
    
    def get_balance(self) -> float:
        """获取余额 Get Balance"""
        return self.__balance
    
    def get_owner(self) -> str:
        return self.__owner
    
    def statement(self):
        """打印账户信息"""
        print(f"=== 账户: {self.__owner} ===")
        print(f"交易记录: {', '.join(self.__transactions)}")
        print(f"当前余额: ${self.__balance:.2f}")

# 使用类
account = BankAccount("Zhang Wei", 1000.0)
account.deposit(500)
account.withdraw(200)
account.deposit(300)
account.withdraw(50)
account.statement()

print(f"\n封装演示 Encapsulation:")
print(f"通过方法访问余额: account.get_balance() = {account.get_balance()}")
print(f"无法直接修改: account.__balance 会报错（属性被隐藏）")

In [ ]:
# === 继承演示 Inheritance Demo ===

class Shape:
    """基类 Base Class"""
    def __init__(self, color: str):
        self.color = color
    
    def area(self) -> float:
        raise NotImplementedError
    
    def describe(self):
        return f"{self.color} {self.__class__.__name__}, area = {self.area():.2f}"

class Circle(Shape):
    def __init__(self, color: str, radius: float):
        super().__init__(color)
        self.radius = radius
    
    def area(self) -> float:
        return 3.14159 * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, color: str, width: float, height: float):
        super().__init__(color)
        self.width = width
        self.height = height
    
    def area(self) -> float:
        return self.width * self.height

class Triangle(Shape):
    def __init__(self, color: str, base: float, height: float):
        super().__init__(color)
        self.base = base
        self.height = height
    
    def area(self) -> float:
        return 0.5 * self.base * self.height

# 多态 Polymorphism
shapes = [
    Circle("Red", 5),
    Rectangle("Blue", 4, 6),
    Triangle("Green", 3, 8)
]

print("=== 多态演示 Polymorphism Demo ===")
print("同一个 describe() 方法，不同的对象有不同的行为：\n")
for shape in shapes:
    print(f"  {shape.describe()}")

In [ ]:
# === 可视化：不同数据类型的内存布局对比 ===

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- 1. 枚举类型 ---
ax = axes[0, 0]
enum_vals = ['SPRING=1', 'SUMMER=2', 'AUTUMN=3', 'WINTER=4']
colors_enum = ['#2ecc71', '#f39c12', '#e67e22', '#3498db']
y_pos = np.arange(len(enum_vals))
ax.barh(y_pos, [1]*4, color=colors_enum, edgecolor='black', height=0.6)
for i, v in enumerate(enum_vals):
    ax.text(0.5, i, v, ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_title('枚举类型 Enumerated\n单一值，有序列表', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1.2)
ax.set_yticks([])
ax.set_xticks([])

# --- 2. 指针类型 ---
ax = axes[0, 1]
ax.add_patch(mpatches.FancyBboxPatch((0.1, 0.6), 0.3, 0.2, boxstyle="round,pad=0.05",
             facecolor='#e74c3c', edgecolor='black', linewidth=2))
ax.text(0.25, 0.7, 'PTR\n0x4A0', ha='center', va='center', fontsize=11, color='white', fontweight='bold')
ax.add_patch(mpatches.FancyBboxPatch((0.6, 0.6), 0.3, 0.2, boxstyle="round,pad=0.05",
             facecolor='#3498db', edgecolor='black', linewidth=2))
ax.text(0.75, 0.7, 'DATA\n42', ha='center', va='center', fontsize=11, color='white', fontweight='bold')
ax.annotate('', xy=(0.6, 0.7), xytext=(0.4, 0.7),
            arrowprops=dict(arrowstyle='->', color='red', lw=3))
ax.text(0.5, 0.4, '\u5730\u5740 0x4A0', ha='center', va='center', fontsize=10, color='gray')
ax.set_title('指针类型 Pointer\n存储地址，间接访问', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0.2, 1)
ax.axis('off')

# --- 3. 记录类型 ---
ax = axes[0, 2]
fields = ['name: "Alice"', 'age: 17', 'grade: "A"', 'enrolled: True']
field_colors = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']
for i, (f, c) in enumerate(zip(fields, field_colors)):
    ax.add_patch(mpatches.FancyBboxPatch((0.1, 0.75 - i*0.2), 0.8, 0.15,
                 boxstyle="round,pad=0.03", facecolor=c, edgecolor='black', linewidth=1.5))
    ax.text(0.5, 0.825 - i*0.2, f, ha='center', va='center', fontsize=11, 
            color='white', fontweight='bold')
ax.set_title('记录类型 Record\n多字段，不同类型', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# --- 4. 集合类型 ---
ax = axes[1, 0]
theta = np.linspace(0, 2*np.pi, 100)
ax.fill(0.5 + 0.35*np.cos(theta), 0.5 + 0.35*np.sin(theta), alpha=0.3, color='#3498db')
ax.plot(0.5 + 0.35*np.cos(theta), 0.5 + 0.35*np.sin(theta), color='#3498db', linewidth=2)
elements = ['A', 'B', 'C', 'D', 'E']
positions = [(0.4, 0.65), (0.6, 0.7), (0.35, 0.45), (0.65, 0.45), (0.5, 0.3)]
for elem, pos in zip(elements, positions):
    ax.text(pos[0], pos[1], elem, ha='center', va='center', fontsize=14, fontweight='bold')
ax.set_title('集合类型 Set\n无序，不重复', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0.05, 0.95)
ax.axis('off')

# --- 5. 类/对象类型 ---
ax = axes[1, 1]
# 类框
ax.add_patch(mpatches.FancyBboxPatch((0.1, 0.1), 0.8, 0.8, boxstyle="round,pad=0.05",
             facecolor='#ecf0f1', edgecolor='#2c3e50', linewidth=3))
# 类名
ax.add_patch(mpatches.FancyBboxPatch((0.1, 0.72), 0.8, 0.18, boxstyle="round,pad=0.05",
             facecolor='#2c3e50', edgecolor='#2c3e50', linewidth=2))
ax.text(0.5, 0.81, 'BankAccount', ha='center', va='center', fontsize=13, 
        color='white', fontweight='bold')
# 属性
ax.text(0.15, 0.62, '- owner: String', fontsize=10)
ax.text(0.15, 0.52, '- balance: Float', fontsize=10)
ax.plot([0.1, 0.9], [0.45, 0.45], color='#2c3e50', linewidth=1.5)
# 方法
ax.text(0.15, 0.37, '+ deposit(amount)', fontsize=10, color='#27ae60')
ax.text(0.15, 0.27, '+ withdraw(amount)', fontsize=10, color='#27ae60')
ax.text(0.15, 0.17, '+ get_balance()', fontsize=10, color='#27ae60')
ax.set_title('类/对象类型 Class\n数据+方法，封装', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# --- 6. 对比总结 ---
ax = axes[1, 2]
ax.axis('off')
table_data = [
    ['类型 Type', '复合?', '有序?', '方法?'],
    ['Enum', 'No', 'Yes', 'No'],
    ['Pointer', 'No', 'N/A', 'No'],
    ['Record', 'Yes', 'N/A', 'No'],
    ['Set', 'Yes', 'No', 'No'],
    ['Class', 'Yes', 'N/A', 'Yes'],
]
table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor('#ecf0f1' if i % 2 == 0 else 'white')
ax.set_title('数据类型对比总结\nComparison Summary', fontsize=12, fontweight='bold')

plt.suptitle('用户自定义数据类型内存布局对比 Memory Layout Comparison', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 三、如何选择合适的数据类型 Choosing the Right Type

| 场景 | 推荐类型 | 原因 |
|:---|:---|:---|
| 有限的固定选项（星期、月份、颜色） | **枚举 Enum** | 类型安全，语义清晰 |
| 动态数据结构（链表、树） | **指针 Pointer** | 需要动态连接节点 |
| 组合不同类型的相关数据 | **记录 Record** | 一个变量包含所有字段 |
| 需要快速查找/去重 | **集合 Set** | O(1)查找，自动去重 |
| 数据+行为，需要继承/封装 | **类 Class** | OOP的完整特性 |

In [ ]:
# === 综合应用案例：学校管理系统 ===

class Subject(Enum):
    """科目（枚举类型）"""
    COMPUTER_SCIENCE = auto()
    MATHEMATICS = auto()
    PHYSICS = auto()
    CHEMISTRY = auto()
    ENGLISH = auto()

class Grade(Enum):
    """成绩等级（枚举类型）"""
    A_STAR = 'A*'
    A = 'A'
    B = 'B'
    C = 'C'
    D = 'D'
    E = 'E'
    U = 'U'

@dataclass
class StudentRecord:
    """学生信息（记录类型 + 集合类型的结合）"""
    name: str
    student_id: str
    subjects: set     # 集合类型：选修的科目
    grades: dict      # 各科成绩

class SchoolSystem:
    """学校系统（类类型）"""
    def __init__(self, school_name: str):
        self.__school_name = school_name
        self.__students = {}  # student_id -> StudentRecord
    
    def enroll(self, student: StudentRecord):
        self.__students[student.student_id] = student
    
    def get_cs_students(self) -> set:
        """获取选修CS的学生（集合运算）"""
        return {s.name for s in self.__students.values() 
                if Subject.COMPUTER_SCIENCE in s.subjects}
    
    def report(self):
        print(f"\n{'='*50}")
        print(f"  {self.__school_name} - Student Report")
        print(f"{'='*50}")
        for sid, s in self.__students.items():
            subj_names = [sub.name for sub in s.subjects]
            print(f"\n  {s.name} ({sid})")
            print(f"  Subjects: {', '.join(subj_names)}")
            for subj, grade in s.grades.items():
                print(f"    {subj.name}: {grade.value}")

# 创建学校系统
school = SchoolSystem("Cambridge International School")

# 注册学生
school.enroll(StudentRecord(
    "Zhang Wei", "S001",
    {Subject.COMPUTER_SCIENCE, Subject.MATHEMATICS, Subject.PHYSICS},
    {Subject.COMPUTER_SCIENCE: Grade.A_STAR, Subject.MATHEMATICS: Grade.A, Subject.PHYSICS: Grade.B}
))
school.enroll(StudentRecord(
    "Li Ming", "S002",
    {Subject.MATHEMATICS, Subject.CHEMISTRY, Subject.ENGLISH},
    {Subject.MATHEMATICS: Grade.A, Subject.CHEMISTRY: Grade.B, Subject.ENGLISH: Grade.A}
))
school.enroll(StudentRecord(
    "Wang Fang", "S003",
    {Subject.COMPUTER_SCIENCE, Subject.ENGLISH},
    {Subject.COMPUTER_SCIENCE: Grade.A, Subject.ENGLISH: Grade.A_STAR}
))

school.report()
print(f"\nCS students: {school.get_cs_students()}")

---

## 练习题 Exercises

### 练习1：概念题

1. 解释为什么用户自定义数据类型是必要的。(4分)
2. 区分复合类型和非复合类型，各举两个例子。(4分)
3. 解释枚举类型相比使用整数常量的优势。(3分)
4. 解释指针类型在实现链表中的作用。(3分)
5. 比较记录类型和类类型的异同。(4分)

### 练习2：编程题

In [ ]:
# 练习2a：创建一个枚举类型 Planet，包含太阳系的8颗行星
# 每颗行星的值是其到太阳的平均距离（百万公里）
# 然后遍历所有行星，打印其名称和距离

# 你的代码：
# class Planet(Enum):
#     MERCURY = 57.9
#     ...


In [ ]:
# 练习2b：创建一个 Library 类，包含：
# - Book 记录类型（title, author, isbn, available）
# - 方法：add_book(), borrow_book(isbn), return_book(isbn), search(keyword)
# - 用集合跟踪所有已借出的ISBN

# 你的代码：


In [ ]:
# 练习2c：实现一个双向链表（Doubly Linked List）
# 每个节点有 data, next（指向下一个节点）, prev（指向上一个节点）
# 实现 append, prepend, display_forward, display_backward 方法

# 你的代码：


### 练习3：考试题风格

**Question 1** (Cambridge style)

A school uses a program to manage student enrollment. The program uses the following data types:

```
TYPE Subject = (Maths, English, Science, History, Geography)

TYPE StudentRecord
    DECLARE StudentID : STRING
    DECLARE Name : STRING
    DECLARE Subjects : SET OF Subject
ENDTYPE
```

(a) Identify the **enumerated type** and explain why it is a non-composite type. [2]

(b) Identify the **composite type** and explain why `StudentRecord` is composite. [2]

(c) Explain one advantage of using an enumerated type for `Subject` rather than a `STRING`. [2]

(d) Explain why a `SET` is used for `Subjects` rather than an `ARRAY`. [2]

---

### 本节小结 Summary

- **非复合类型**不由其他类型组合而成：枚举（有限有序值列表）、指针（存储地址）
- **复合类型**由多种类型组合而成：记录（多字段）、集合（无序不重复）、类（数据+方法）
- 选择数据类型时要考虑：语义清晰性、类型安全性、操作需求、性能需求